In [ ]:
from pathlib import Path
import random
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.cluster import AgglomerativeClustering
from ultralytics import YOLO

# 1. Проверка GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Вычислительное устройство: {device}")
if device.type == "cuda":
    print(f"Видеокарта: {torch.cuda.get_device_name(0)}")

# 2. Загрузка детектора
BASE_DIR = Path(r"D:\Tabmed\Tabmed_ml")
MODEL_PATH = BASE_DIR / "best.pt"
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Веса не найдены: {MODEL_PATH}")

yolo = YOLO(str(MODEL_PATH))
model_backbone = yolo.model.to(device)
model_backbone.eval()

# Слой адаптивного пулинга для карт признаков бэкбона
pool = nn.AdaptiveAvgPool2d((1, 1)).to(device)

print("Модель детектора и бэкбон готовы к инференсу.")

In [ ]:
# Папка со снимками
IMAGES_DIR = Path(r"D:\Tabmed\Datasets\dataset1\images")
if not IMAGES_DIR.exists():
    IMAGES_DIR = BASE_DIR

# Собираем все доступные фото
all_image_paths = sorted(
    list(IMAGES_DIR.glob("*.jpg")) + list(IMAGES_DIR.glob("*.png"))
)

# Выбираем для теста до 8 кадров
random.seed(42)
sample_paths = random.sample(all_image_paths, min(8, len(all_image_paths)))
print(f"Выбрано фото для пакетного анализа: {len(sample_paths)}")

# 1. Инференс детектора и сохранение результатов
batch_results = []
for p in sample_paths:
    res = yolo.predict(source=str(p), conf=0.35, verbose=False)[0]
    batch_results.append((p, res))

# 2. Отрисовка сетки кадров с боксами
n_imgs = len(batch_results)
cols = 2
rows = (n_imgs + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 6))
if rows == 1 and cols == 1:
    axes = np.array([axes])
axes = np.array(axes).flatten()

for idx, (p, res) in enumerate(batch_results):
    annotated = res.plot(line_width=3, font_size=1.0)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    # Сжатие превью для плавного отображения без зависаний
    h, w = annotated_rgb.shape[:2]
    scale = 700 / max(h, w)
    preview = cv2.resize(
        annotated_rgb,
        (int(w * scale), int(h * scale)),
        interpolation=cv2.INTER_AREA,
    )

    n_det = len(res.boxes)
    axes[idx].imshow(preview)
    axes[idx].set_title(
        f"{p.name} | Найдено объектов: {n_det}", fontsize=11, fontweight="bold"
    )
    axes[idx].axis("off")

# Отключаем пустые слоты
for ax in axes[n_imgs:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics.pairwise import cosine_distances

# Расчет матрицы расстояний между всеми 37 кропами
dists = cosine_distances(feats_norm)
triu_dists = dists[np.triu_indices_from(dists, k=1)]

print(f"Минимальное расстояние: {triu_dists.min():.4f}")
print(f"Среднее расстояние:     {triu_dists.mean():.4f}")
print(f"Медианное расстояние:   {np.median(triu_dists):.4f}")
print(f"Максимальное расстояние:{triu_dists.max():.4f}")

In [ ]:
from sklearn.metrics.pairwise import cosine_distances

all_crops_data = []
crop_global_id = 0
PADDING_RATIO = 0.08

# 1. Нарезка кропов
for p, res in batch_results:
    orig_bgr = cv2.imread(str(p))
    if orig_bgr is None:
        continue
    h_img, w_img = orig_bgr.shape[:2]

    boxes = res.boxes.xyxy.cpu().numpy()
    classes = res.boxes.cls.cpu().numpy()
    confs = res.boxes.conf.cpu().numpy()

    for b, c, cf in zip(boxes, classes, confs):
        x1, y1, x2, y2 = b
        bw, bh = x2 - x1, y2 - y1
        pw, ph = int(bw * PADDING_RATIO), int(bh * PADDING_RATIO)

        cx1, cy1 = max(0, int(x1 - pw)), max(0, int(y1 - ph))
        cx2, cy2 = min(w_img, int(x2 + pw)), min(h_img, int(y2 + ph))

        crop = orig_bgr[cy1:cy2, cx1:cx2]
        if crop.size == 0:
            continue

        all_crops_data.append(
            {
                "crop_id": crop_global_id,
                "source_image": p.name,
                "class_id": int(c),
                "class_name": "capsule" if int(c) == 0 else "tablet",
                "conf": float(cf),
                "crop_bgr": crop,
            }
        )
        crop_global_id += 1

print(f"Всего вырезано кропов: {len(all_crops_data)}")

# 2. Извлечение признаков бэкбоном
IMG_SIZE = 128
tensors = []
for it in all_crops_data:
    resized = cv2.resize(it["crop_bgr"], (IMG_SIZE, IMG_SIZE))
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    t = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0
    tensors.append(t)

batch_tensor = torch.stack(tensors).to(device)

with torch.no_grad():
    x = batch_tensor
    for layer in model_backbone.model[:10]:
        x = layer(x)
    feats = pool(x).flatten(1).cpu().numpy()

# 3. Устранение анизотропии: Вычитание среднего (Mean Subtraction) + L2
feats_centered = feats - feats.mean(axis=0, keepdims=True)
norms = np.linalg.norm(feats_centered, axis=1, keepdims=True)
norms[norms == 0] = 1e-10
feats_norm = feats_centered / norms

# 4. Построение матрицы косинусных расстояний
dist_matrix = cosine_distances(feats_norm)

# Жесткий фильтр детектора: капсулы и таблетки разводим бесконечным расстоянием
for i in range(len(all_crops_data)):
    for j in range(i + 1, len(all_crops_data)):
        if all_crops_data[i]["class_id"] != all_crops_data[j]["class_id"]:
            dist_matrix[i, j] = 2.0
            dist_matrix[j, i] = 2.0

# 5. Кластеризация по предварительно вычисленной матрице
# Для центрированных признаков оптимальный порог обычно 0.60 - 0.75
CLUSTERING_THRESHOLD = 0.65

clusterer = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=CLUSTERING_THRESHOLD,
    metric="precomputed",
    linkage="average",
)
labels = clusterer.fit_predict(dist_matrix).tolist()

for it, lbl in zip(all_crops_data, labels):
    it["cluster_id"] = int(lbl)

unique_clusters = sorted(list(set(labels)))
print(f"Сформировано кластеров: {len(unique_clusters)}")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

CROP_SIZE = 160        # Единый размер плитки (Ш x В)
LABEL_HEIGHT = 45      # Высота информационной плашки под кропом
MAX_PER_ROW = 8        # Количество кропов в одной сплошной строке

full_canvas_blocks = []

for cid in unique_clusters:
    group_items = [it for it in all_crops_data if it["cluster_id"] == cid]
    classes = [it["class_name"] for it in group_items]
    dominant_class = max(set(classes), key=classes.count)
    n_items = len(group_items)

    # 1. Заголовок кластера
    row_width = min(n_items, MAX_PER_ROW) * CROP_SIZE
    header = np.full((40, row_width, 3), 40, dtype=np.uint8)
    title_text = f"Кластер {cid} | Тип: {dominant_class} | Всего: {n_items} шт."
    cv2.putText(header, title_text, (15, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
    full_canvas_blocks.append(header)

    # 2. Подготовка плиток кропов
    tiles = []
    for it in group_items:
        # Ресайз самого кропа в аккуратный квадрат
        tile_img = cv2.resize(it["crop_bgr"], (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_AREA)

        # Нижняя информационная плашка
        banner = np.full((LABEL_HEIGHT, CROP_SIZE, 3), 20, dtype=np.uint8)
        line1 = f"ID:{it['crop_id']} | {it['class_name']}"
        line2 = f"conf:{it['conf']:.2f}"

        cv2.putText(banner, line1, (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (220, 220, 220), 1, cv2.LINE_AA)
        cv2.putText(banner, line2, (6, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (0, 215, 255), 1, cv2.LINE_AA)

        # Тонкая рамка (1 px) вокруг плитки для четкого разделения таблеток
        combined_tile = np.vstack([tile_img, banner])
        cv2.rectangle(combined_tile, (0, 0), (CROP_SIZE - 1, CROP_SIZE + LABEL_HEIGHT - 1), (60, 60, 60), 1)
        tiles.append(combined_tile)

    # 3. Сборка строк без единого разрыва
    for row_start in range(0, n_items, MAX_PER_ROW):
        row_tiles = tiles[row_start : row_start + MAX_PER_ROW]
        
        # Если в последней строке плиток меньше, дополняем нейтральным фоном
        needed = MAX_PER_ROW - len(row_tiles)
        if needed > 0 and n_items > MAX_PER_ROW:
            empty_tile = np.full((CROP_SIZE + LABEL_HEIGHT, CROP_SIZE, 3), 30, dtype=np.uint8)
            row_tiles.extend([empty_tile] * needed)

        row_concat = np.hstack(row_tiles)
        full_canvas_blocks.append(row_concat)

    # Небольшой разделитель между разными кластерами
    sep = np.full((12, row_width if n_items <= MAX_PER_ROW else MAX_PER_ROW * CROP_SIZE, 3), 15, dtype=np.uint8)
    full_canvas_blocks.append(sep)

# Выравниваем блоки по максимальной ширине, если кластеры разной длины
max_w = max(block.shape[1] for block in full_canvas_blocks)
aligned_blocks = []
for b in full_canvas_blocks:
    if b.shape[1] < max_w:
        pad = np.full((b.shape[0], max_w - b.shape[1], 3), 15, dtype=np.uint8)
        aligned_blocks.append(np.hstack([b, pad]))
    else:
        aligned_blocks.append(b)

# 4. Итоговая монолитная картинка
seamless_canvas = np.vstack(aligned_blocks)
canvas_rgb = cv2.cvtColor(seamless_canvas, cv2.COLOR_BGR2RGB)

# Отрисовка одним цельным холстом
plt.figure(figsize=(14, max(4, int(seamless_canvas.shape[0] / 80))))
plt.imshow(canvas_rgb)
plt.axis("off")
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.show()



In [ ]:
# Находим индексы объектов каждого проблемного кластера
idx_c1 = [i for i, it in enumerate(all_crops_data) if it["crop_id"] in [23, 32]]
idx_c3 = [i for i, it in enumerate(all_crops_data) if it["crop_id"] in [24, 25, 26, 27, 28, 29, 30, 31]]
idx_c4 = [i for i, it in enumerate(all_crops_data) if it["crop_id"] in [15, 16, 17, 18, 19, 20, 21, 22]]

# Среднее расстояние между Кластером 1 и Кластером 3 (одни и те же таблетки)
d_1_to_3 = dist_matrix[np.ix_(idx_c1, idx_c3)].mean()

# Среднее расстояние между коричневыми (Кластер 3) и бежевыми (Кластер 4) таблетками
d_3_to_4 = dist_matrix[np.ix_(idx_c3, idx_c4)].mean()

print(f"Расстояние между Кластером 1 и Кластером 3 (коричневые между собой): {d_1_to_3:.4f}")
print(f"Расстояние между Кластером 3 и Кластером 4 (коричневые vs бежевые):    {d_3_to_4:.4f}")
print(f"Текущий порог: 0.65")